# Cloud Misconfiguration Detection Using Machine Learning
### End-to-End Experimental Pipeline: Feature Extraction, GNN Security Graph, RF Triage, Intent NLP & Compiler Remediation
**Authors:** Group 8 (Israt Jahan Rani - 011201376, Robiul Islam Tamim - 011212039)

In [1]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print(f"PyTorch Version: {torch.__version__}")

## 1. Load Extracted Features and Benchmark Metadata
The dataset consists of 1,100 multi-domain manifests (600 Kubernetes, 300 IAM/Storage, 200 Serverless SAM).

In [2]:
data_path = Path('../data/labeled/features_dataset.npz')
meta_path = Path('../data/labeled/features_metadata.csv')

data = np.load(data_path)
X, y = data['X'], data['y']
df_meta = pd.read_csv(meta_path)

print(f"Feature matrix shape: {X.shape}")
print(f"Labels count:\n{df_meta['label'].value_counts()}\n")
print(f"Domain breakdown:\n{df_meta['domain'].value_counts()}")

## 2. Stage 3A: Calibrated Random Forest Triage
Evaluates 880 training and 220 testing samples using 500 trees and probability calibration.

In [3]:
import joblib
model_rf = joblib.load('../models/rf_calibrated.pkl')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
probas = model_rf.predict_proba(X_test)[:, 1]
preds = (probas >= 0.5).astype(int)

print("=== Random Forest Test Classification Report ===")
print(classification_report(y_test, preds, target_names=['COMPLIANT', 'MISCONFIGURED'], digits=4))
print(f"ROC-AUC: {roc_auc_score(y_test, probas):.4f}")

## 3. Stage 2: Heterogeneous Security Graph & GNN
Evaluates cross-resource threat paths (Pod -> IAM Role -> Storage Bucket).

In [4]:
from src.stage2_graph.gnn_model import SecurityHeteroGNN
from src.stage2_graph.graph_builder import build_hetero_security_graph

graph = build_hetero_security_graph(X, y, df_meta)
model_gnn = SecurityHeteroGNN()
model_gnn.load_state_dict(torch.load('../models/gnn_security.pt', weights_only=True))
model_gnn.eval()

print(f"GNN Loaded successfully. Graph contains {sum(len(v) for v in graph.node_features.values())} nodes.")

## 4. Stage 5: NLP Intent Classifier
Suppresses false alarms on intentional public resources.

In [5]:
intent_pipe = joblib.load('../models/intent_classifier/intent_pipeline.pkl')
test_sample = "name: website-static-assets | labels: app:frontend env:prod | comments: public website cdn"
pred_intent = intent_pipe.predict([test_sample])[0]
print(f"Sample comment: '{test_sample}'")
print(f"Predicted Intent: {pred_intent}")